# You are bot — решение

Задача: по диалогу определить, является ли каждый из двух участников ботом. Метрика — LogLoss по вероятностям (чем меньше, тем лучше).

**Решение — одна модель LightGBM**, та же логика, что в сервисе `bot_detector`. Каждый участник диалога — одна строка. Признаки считаются для участника и для его собеседника:
1. число и длина сообщений;
2. статистики текста (пунктуация, заглавные, латиница, вопросы, повторы);
3. признаки из эмбеддингов замороженного `rubert-tiny2`: средний смысл реплик и то, насколько реплики отвечают по теме собеседнику.

Качество на кросс-валидации (5 фолдов по диалогам): **LogLoss 0.200**; ROC-AUC 0.976, F1 0.90 при пороге 0.5.

План ноутбука:
1. Данные и первичный взгляд
2. Признаки
3. Модель
4. Метрики классификации
5. Сабмит

Запуск: данные в `data/` в корне проекта, затем «Run All». Работает на CPU: эмбеддинги считаются меньше минуты, весь ноутбук — пара минут. В конце сохраняется `submissions/submission.csv`.

In [41]:
import json
import os
import subprocess
import sys
from pathlib import Path

import lightgbm as lgb
import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.metrics import f1_score, log_loss, precision_score, recall_score, roc_auc_score
from sklearn.model_selection import GroupKFold, cross_val_predict
from sklearn.pipeline import Pipeline

# ноутбук лежит в notebooks/, а пути к данным и кэшу — от корня проекта
if os.path.basename(os.getcwd()) == 'notebooks':
    os.chdir('..')

SEED = 42
N_SPLITS = 5
CLIP = 0.001

## 1. Данные и первичный взгляд

`train.json` / `test.json` — словарь `{dialog_id: [сообщения]}`, у сообщения есть `text`, порядковый номер `message` и `participant_index` (0 или 1). Метки `is_bot` — в `ytrain.csv` для каждой пары (диалог, участник).

Разворачиваем диалоги в таблицу сообщений, затем собираем одну строку на участника: все его реплики одним текстом, число сообщений и средняя длина.

In [42]:
def load_messages(path):
    with open(path, encoding='utf-8') as f:
        dialogs = json.load(f)
    rows = [
        {'dialog_id': dialog_id, **msg}
        for dialog_id, messages in dialogs.items()
        for msg in messages
    ]
    return pd.DataFrame(rows)

train_msgs = load_messages('data/train.json')
test_msgs = load_messages('data/test.json')
print('Сообщений:', len(train_msgs), len(test_msgs))
train_msgs.head()

Сообщений: 10490 2837


,dialog_id,text,message,participant_index
0,b5b8e0ae-5516-4150-9004-00fe422d5e33,"how can i helpt you, men?",0,0
1,b5b8e0ae-5516-4150-9004-00fe422d5e33,Никак,1,1
2,b5b8e0ae-5516-4150-9004-00fe422d5e33,неа,2,0
3,b5b8e0ae-5516-4150-9004-00fe422d5e33,лох,3,1
4,b5b8e0ae-5516-4150-9004-00fe422d5e33,не тупи,4,0


In [43]:
def aggregate(msgs):
    msgs = msgs.sort_values(['dialog_id', 'message'])
    return (
        msgs.groupby(['dialog_id', 'participant_index'])
        .agg(
            text=('text', lambda s: ' \n '.join(s)),
            n_messages=('text', 'size'),
            mean_len=('text', lambda s: s.str.len().mean()),
        )
        .reset_index()
    )

ytrain = pd.read_csv('data/ytrain.csv')
ytest = pd.read_csv('data/ytest.csv')

# how='left' от файла с метками: участник без сообщений всё равно остаётся в выборке
train = ytrain.merge(aggregate(train_msgs), on=['dialog_id', 'participant_index'], how='left')
test = ytest.merge(aggregate(test_msgs), on=['dialog_id', 'participant_index'], how='left')
train = train.fillna({'text': '', 'n_messages': 0, 'mean_len': 0})
test = test.fillna({'text': '', 'n_messages': 0, 'mean_len': 0})

print('Участников:', train.shape[0], test.shape[0])
print('Доля ботов в train:', round(train['is_bot'].mean(), 3))
print('Ботов в диалоге:', train.groupby('dialog_id')['is_bot'].sum().value_counts().to_dict())

Участников: 3032 758
Доля ботов в train: 0.408
Ботов в диалоге: {1: 1237, 0: 279}


Доля ботов и длина реплик у ботов и людей.

In [44]:
train['is_bot'].value_counts(normalize=True)

is_bot
0    0.592018
1    0.407982
Name: proportion, dtype: float64

In [45]:
train.groupby('is_bot')[['n_messages', 'mean_len']].describe()

n_messages                                               mean_len  \
            count      mean       std  min  25%  50%  75%   max    count   
is_bot                                                                     
0          1795.0  3.098607  2.738668  1.0  2.0  3.0  4.0  68.0   1795.0   
1          1237.0  3.983832  3.239081  1.0  2.0  3.0  5.0  69.0   1237.0   

                                                                         
             mean         std  min       25%        50%   75%       max  
is_bot                                                                   
0       30.885066  513.023267  1.0  5.666667   8.666667  13.5  16458.75  
1       32.903285   88.090114  1.0  7.000000  11.666667  28.0   1903.50

## 2. Признаки

### 2.1. Собеседник
В каждом диалоге ровно два участника, поэтому собеседник — строка того же диалога с `participant_index = 1 - participant_index`. Его реплики сильно помогают: человек, говорящий с ботом, часто его проверяет («ты бот?»), а бот отвечает развёрнуто и по теме.

In [46]:
def add_partner(df):
    partner = (
        df[['dialog_id', 'participant_index', 'text', 'n_messages', 'mean_len']]
        .assign(participant_index=lambda d: 1 - d['participant_index'])
        .rename(columns={'text': 'partner_text', 'n_messages': 'partner_n_messages', 'mean_len': 'partner_mean_len'})
    )
    return df.merge(partner, on=['dialog_id', 'participant_index'], how='left')

train = add_partner(train)
test = add_partner(test)

train

,dialog_id,participant_index,is_bot,text,n_messages,mean_len,partner_text,partner_n_messages,partner_mean_len
0,0012a424-ae56-414b-a982-04cf24c35229,0,1,Экономическая ситуация в Зимбабве за последнее...,3,172.000000,ух ты \n про себя?,2,7.000000
1,0012a424-ae56-414b-a982-04cf24c35229,1,0,ух ты \n про себя?,2,7.000000,Экономическая ситуация в Зимбабве за последнее...,3,172.000000
2,009a6b25-0289-4845-a392-4025fde96371,0,1,answer \n answer \n answer \n answer \n answer...,7,6.000000,"Привет \n Грустно, что ты не человек \n Любишь...",6,11.166667
3,009a6b25-0289-4845-a392-4025fde96371,1,0,"Привет \n Грустно, что ты не человек \n Любишь...",6,11.166667,answer \n answer \n answer \n answer \n answer...,7,6.000000
4,00ade168-91d4-4605-b99b-da1ecbbb51fd,0,0,ку,1,2.000000,"привет! я gigachat, чем могу помочь?",1,36.000000
...,...,...,...,...,...,...,...,...,...
3027,ff7e18e6-ae48-4d2b-b390-85a4a1ac7897,1,1,гойда \n америка \n нефор не любит россию,3,11.000000,Привет! \n Фу( \n Россия!,3,5.666667
3028,ffc0a18e-7ee9-4a0c-8f36-4966f24e33c4,0,0,123,1,3.000000,привет кринж \n 123 123,2,9.500000
3029,ffc0a18e-7ee9-4a0c-8f36-4966f24e33c4,1,1,привет кринж \n 123 123,2,9.500000,123,1,3.000000
3030,ffdd69d6-8e2d-4940-942f-2841209aa736,0,0,What?,1,5.000000,What?,1,5.000000


### 2.2. Статистики текста
Доли латиницы, заглавных и пунктуации, частота вопросов и скобочек-смайлов, длина в словах, самая длинная реплика и доля уникальных реплик. Текст специально не чистим: эмодзи, markdown, регистр и опечатки — это и есть сигнал «бот / человек».

In [47]:
stat_cols = ['latin', 'upper', 'punct', 'question', 'emoji', 'mean_words', 'max_len', 'unique_ratio']

def text_stats(msgs):
    s = msgs['text']
    length = s.str.len().clip(lower=1)
    return (
        msgs.assign(
            latin=s.str.count(r'[A-Za-z]') / length,
            upper=s.str.count(r'[A-ZА-ЯЁ]') / length,
            punct=s.str.count(r'[.,!?;:]') / length,
            question=s.str.contains(r'\?').astype(int),
            emoji=s.str.contains(r'[)(]').astype(int),
            n_words=s.str.split().str.len(),
        )
        .groupby(['dialog_id', 'participant_index'])
        .agg(
            latin=('latin', 'mean'),
            upper=('upper', 'mean'),
            punct=('punct', 'mean'),
            question=('question', 'mean'),
            emoji=('emoji', 'mean'),
            mean_words=('n_words', 'mean'),
            max_len=('text', lambda t: t.str.len().max()),
            unique_ratio=('text', lambda t: t.nunique() / len(t)),
        )
        .reset_index()
    )

def add_stats(df, msgs):
    stats = text_stats(msgs)
    partner = (
        stats.assign(participant_index=lambda d: 1 - d['participant_index'])
        .rename(columns={c: 'partner_' + c for c in stat_cols})
    )
    return (
        df.merge(stats, on=['dialog_id', 'participant_index'], how='left')
        .merge(partner, on=['dialog_id', 'participant_index'], how='left')
    )

train = add_stats(train, train_msgs)
test = add_stats(test, test_msgs)
train.groupby('is_bot')[stat_cols].mean().T

is_bot,0,1
latin,0.230058,0.188143
upper,0.077150,0.048325
punct,0.038136,0.036796
question,0.198912,0.290288
emoji,0.023555,0.047818
mean_words,2.552303,5.284037
max_len,75.994986,63.772029
unique_ratio,0.963231,0.952789


### 2.3. Эмбеддинги rubert-tiny2
Каждое сообщение кодируем **замороженным** `cointegrated/rubert-tiny2` (29M параметров, вектор из 312 чисел) — модель не дообучается. Векторы нормированы, поэтому скалярное произведение — косинусная близость. Признаки участника:
- `coh_mean / coh_min / coh_max` — близость каждой его реплики к предыдущей реплике собеседника: отвечает ли он по теме;
- `self_sim_mean` — близость соседних собственных реплик: повторяется ли он;
- `emb_0 … emb_311` — средний эмбеддинг его реплик.

Те же признаки собеседника — с префиксом `partner_`. Если перед репликами участника не было реплик собеседника, `coh_*` пустые — LightGBM умеет с этим работать.

Эмбеддинги считает `scripts/embed.py` отдельным процессом и сохраняет в `cache/`: на macOS torch и lightgbm в одном процессе роняют ядро. Векторы считаются один раз; чтобы пересчитать, удалите `cache/`.

In [48]:
if not (Path('cache/emb_train.npy').exists() and Path('cache/emb_test.npy').exists()):
    print(subprocess.run([sys.executable, 'scripts/embed.py'], check=True, capture_output=True, text=True).stdout)

train_emb = np.load('cache/emb_train.npy')
test_emb = np.load('cache/emb_test.npy')
assert len(train_emb) == len(train_msgs) and len(test_emb) == len(test_msgs)
train_emb.shape, test_emb.shape

((10490, 312), (2837, 312))

In [49]:
def embedding_features(msgs, emb):
    rows = []
    for dialog_id, d in msgs.reset_index(drop=True).groupby('dialog_id', sort=False):
        d = d.sort_values('message')
        who = d['participant_index'].to_numpy()
        e = emb[d.index.to_numpy()]
        for p in (0, 1):
            own = np.flatnonzero(who == p)
            coh, self_sim = [], []
            for k, i in enumerate(own):
                prev_partner = np.flatnonzero(who[:i] != p)
                if len(prev_partner):
                    coh.append(e[i] @ e[prev_partner[-1]])
                if k > 0:
                    self_sim.append(e[i] @ e[own[k - 1]])
            mean_emb = e[own].mean(axis=0) if len(own) else np.zeros(emb.shape[1])
            rows.append({
                'dialog_id': dialog_id,
                'participant_index': p,
                'coh_mean': np.mean(coh) if coh else np.nan,
                'coh_min': np.min(coh) if coh else np.nan,
                'coh_max': np.max(coh) if coh else np.nan,
                'self_sim_mean': np.mean(self_sim) if self_sim else np.nan,
                **{f'emb_{j}': v for j, v in enumerate(mean_emb)},
            })
    return pd.DataFrame(rows)

def add_embedding_features(df, msgs, emb):
    feats = embedding_features(msgs, emb)
    cols = [c for c in feats.columns if c not in ('dialog_id', 'participant_index')]
    partner = (
        feats.assign(participant_index=lambda d: 1 - d['participant_index'])
        .rename(columns={c: 'partner_' + c for c in cols})
    )
    return (
        df.merge(feats, on=['dialog_id', 'participant_index'], how='left')
        .merge(partner, on=['dialog_id', 'participant_index'], how='left')
    )

train = add_embedding_features(train, train_msgs, train_emb)
test = add_embedding_features(test, test_msgs, test_emb)

coh_cols = ['coh_mean', 'coh_min', 'coh_max', 'self_sim_mean']
emb_cols = [c for c in train.columns if c.startswith('emb_')]
train.groupby('is_bot')[coh_cols + ['partner_' + c for c in coh_cols]].mean().T

is_bot,0,1
coh_mean,0.717453,0.806020
coh_min,0.646875,0.741653
coh_max,0.785687,0.864868
self_sim_mean,0.751984,0.750168
partner_coh_mean,0.779386,0.717054
partner_coh_min,0.716514,0.641407
partner_coh_max,0.837949,0.789990
partner_self_sim_mean,0.740893,0.767279


## 3. Модель

LightGBM на всех признаках сразу: 652 числа на участника. Гиперпараметры подобраны Optuna — код ниже.

**Валидация** — `GroupKFold` по `dialog_id`: оба участника одного диалога всегда в одном фолде, иначе модель видела бы собеседника на обучении и оценка была бы завышена. `cross_val_predict` даёт для каждого участника train предсказание модели, которая его не видела (OOF).

In [54]:
features = (
    ['n_messages', 'mean_len', 'partner_n_messages', 'partner_mean_len']
    + stat_cols + ['partner_' + c for c in stat_cols]
    + coh_cols + ['partner_' + c for c in coh_cols]
    + emb_cols + ['partner_' + c for c in emb_cols]
)
print('Признаков:', features)

model = Pipeline([
    ('features', ColumnTransformer([('num', 'passthrough', features)])),
    ('clf', lgb.LGBMClassifier(
        n_estimators=700, learning_rate=0.024, num_leaves=22, min_child_samples=18,
        subsample=0.94, subsample_freq=1, colsample_bytree=0.5, reg_lambda=0.53,
        random_state=SEED, verbose=-1,
    )),
])

cv = GroupKFold(n_splits=N_SPLITS)
y = train['is_bot'].to_numpy()

oof = cross_val_predict(
    model, train[features], y,
    groups=train['dialog_id'], cv=cv, method='predict_proba',
)[:, 1]
print('CV LogLoss:', round(log_loss(y, oof), 4))
print('Константа (доля ботов):', round(log_loss(y, np.full(len(y), y.mean())), 4))

Признаков: ['n_messages', 'mean_len', 'partner_n_messages', 'partner_mean_len', 'latin', 'upper', 'punct', 'question', 'emoji', 'mean_words', 'max_len', 'unique_ratio', 'partner_latin', 'partner_upper', 'partner_punct', 'partner_question', 'partner_emoji', 'partner_mean_words', 'partner_max_len', 'partner_unique_ratio', 'coh_mean', 'coh_min', 'coh_max', 'self_sim_mean', 'partner_coh_mean', 'partner_coh_min', 'partner_coh_max', 'partner_self_sim_mean', 'emb_0', 'emb_1', 'emb_2', 'emb_3', 'emb_4', 'emb_5', 'emb_6', 'emb_7', 'emb_8', 'emb_9', 'emb_10', 'emb_11', 'emb_12', 'emb_13', 'emb_14', 'emb_15', 'emb_16', 'emb_17', 'emb_18', 'emb_19', 'emb_20', 'emb_21', 'emb_22', 'emb_23', 'emb_24', 'emb_25', 'emb_26', 'emb_27', 'emb_28', 'emb_29', 'emb_30', 'emb_31', 'emb_32', 'emb_33', 'emb_34', 'emb_35', 'emb_36', 'emb_37', 'emb_38', 'emb_39', 'emb_40', 'emb_41', 'emb_42', 'emb_43', 'emb_44', 'emb_45', 'emb_46', 'emb_47', 'emb_48', 'emb_49', 'emb_50', 'emb_51', 'emb_52', 'emb_53', 'emb_54', 'emb

### 3.1. Подбор гиперпараметров

Optuna, 60 итераций по той же кросс-валидации. Параметры выше найдены этим поиском; он идёт долго, поэтому по умолчанию выключен.

In [51]:
RUN_TUNING = False

if RUN_TUNING:
    import optuna

    X = train[features].to_numpy()

    def objective(trial):
        params = dict(
            n_estimators=trial.suggest_int('n_estimators', 200, 1500, step=100),
            learning_rate=trial.suggest_float('learning_rate', 0.01, 0.1, log=True),
            num_leaves=trial.suggest_int('num_leaves', 4, 63, log=True),
            min_child_samples=trial.suggest_int('min_child_samples', 5, 60),
            subsample=trial.suggest_float('subsample', 0.5, 1.0),
            colsample_bytree=trial.suggest_float('colsample_bytree', 0.05, 0.6, log=True),
            reg_lambda=trial.suggest_float('reg_lambda', 1e-3, 30, log=True),
        )
        tuned_oof = np.zeros(len(y))
        for tr, va in cv.split(X, y, groups=train['dialog_id']):
            clf = lgb.LGBMClassifier(**params, subsample_freq=1, random_state=SEED, verbose=-1)
            clf.fit(X[tr], y[tr])
            tuned_oof[va] = clf.predict_proba(X[va])[:, 1]
        return log_loss(y, tuned_oof)

    study = optuna.create_study(direction='minimize', sampler=optuna.samplers.TPESampler(seed=SEED))
    study.optimize(objective, n_trials=60, show_progress_bar=True)
    print(study.best_value, study.best_params)

## 4. Метрики классификации

Для справки — метрики по меткам при разных порогах. Лидерборд считается по LogLoss, порог на сабмит не влияет.

In [52]:
print('ROC-AUC:', round(roc_auc_score(y, oof), 4))
pd.DataFrame([
    {
        'порог': t,
        'F1': f1_score(y, oof >= t),
        'precision': precision_score(y, oof >= t),
        'recall': recall_score(y, oof >= t),
    }
    for t in [0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8]
]).round(3)

ROC-AUC: 0.9757


,порог,F1,precision,recall
0,0.2,0.895,0.853,0.942
1,0.3,0.903,0.884,0.922
2,0.4,0.904,0.905,0.903
3,0.5,0.904,0.925,0.884
4,0.6,0.896,0.939,0.857
5,0.7,0.886,0.953,0.828
6,0.8,0.866,0.967,0.784


## 5. Сабмит

Обучаем модель на всём train и предсказываем test. Вероятности обрезаем до [0.001, 0.999]: уверенная ошибка с вероятностью около 0 или 1 дала бы огромный LogLoss.

In [53]:
model.fit(train[features], y)
preds = model.predict_proba(test[features])[:, 1]

submission = test[['ID']].assign(is_bot=np.clip(preds, CLIP, 1 - CLIP))
assert len(submission) == len(ytest)
assert (submission['ID'] == ytest['ID']).all()
submission.to_csv('submissions/submission.csv', index=False)
submission.head()

,ID,is_bot
0,0253c2df-7cea-4456-85d1-35f776c4f671_0,0.010110
1,0253c2df-7cea-4456-85d1-35f776c4f671_1,0.034525
2,03641877-db32-43b1-b78a-fba5a4aafa2d_0,0.987495
3,03641877-db32-43b1-b78a-fba5a4aafa2d_1,0.015992
4,0396d8a8-6f1b-437b-860e-2837683cb555_0,0.097218
